In [1]:
arqs <- c("qrels_Andre_2026-10-25.csv")

In [2]:
estado()

ERROR: Error in estado(): could not find function "estado"


In [3]:
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"

source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor06.R")

download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R")
source("config.R")

cp <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"),
               stringsAsFactors = FALSE, fileEncoding = "UTF-8")

docs <- setNames(cp$texto, cp$id)

origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))

ix <- montar(docs, cfg)

estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 18:59 · R version 4.6.1 (2026-06-24) · motor06 v2 2026-10-08 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `arqs` | character | 1 | qrels_Andre_2026-10-25.csv |
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 12 | de, e, a, o, em, da, ... |
| `cfg$grupo` | character | 1 | Predicarga |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 1 |
| `cp` | data.frame | 64 × 5 | colunas: id, titulo, texto, fonte, url |
| `docs` | character, nomeado | 64 | d1 São Vicente, oficialmente E..., d2 São Vicente marca o início ..., d3 Com um PIB de mais de 6 bil..., d4 São Vicente preserva divers..., d5 A Ilha de Gohayó foi descob..., d6 Nas primeiras décadas do sé..., ... |
| `ix$tokens` | list | 64 | el

In [4]:
arqs <- c("qrels_Andre_2026-10-25.csv")

In [8]:
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 19:02 · R version 4.6.1 (2026-06-24) · motor06 v2 2026-10-08 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `a` | character | 1 | qrels_Andre_2026-10-25.csv |
| `arqs` | character | 1 | qrels_Andre_2026-10-25.csv |
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 12 | de, e, a, o, em, da, ... |
| `cfg$grupo` | character | 1 | Predicarga |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 1 |
| `consultas` | character, nomeado | 5 | q01 monumentos historicos centro, q02 fundacao vila martim afonso, q03 ponte pensil arquitetura, q04 ataques piratas baixada san..., q05 primeiros habitantes coloni... |
| `cp` | data.frame | 64 × 5 | colunas: id, titulo, texto, fonte, url |
| `docs` | characte

In [6]:
arqs <- c("qrels_Andre_2026-10-25.csv")

for (a in arqs)
  download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)

qrels <- ler_qrels(arqs, juiz = cfg$juizes)

nec <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"),
                stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")

consultas <- setNames(nec$texto_consulta, nec$consulta)

R_q <- sapply(qrels, function(g) sum(g >= cfg$limiar))

usar <- names(qrels)[R_q > 0]

names(qrels)[R_q == 0]

character(0)

In [7]:
names(qrels)[R_q == 0]

character(0)

In [9]:
names(qrels)
usar
R_q

[1] "q01" "q02" "q03" "q04" "q05"

[1] "q01" "q02" "q03" "q04" "q05"

q01 q02 q03 q04 q05 
  1   5   2   6   6

In [10]:
# Rocchio: reformula a consulta a partir dos relevantes e nao relevantes
rocchio <- function(q, Dr, Dnr = character(0), ix, a = 1, b = 0.75, g = 0.15) {
  cr  <- rowMeans(ix$w[, Dr, drop = FALSE])       # centroide dos documentos relevantes
  cnr <- if (length(Dnr)) rowMeans(ix$w[, Dnr, drop = FALSE]) else 0  # centroide dos nao relevantes; se vazio, 0
  a * q + b * cr - g * cnr                        # formula: consulta + relevantes - nao relevantes
}

# Ranking vetorial: cosseno entre a consulta reformulada e cada documento
ranking_vetor <- function(q, ix)
  sort(apply(ix$w, 2, function(d) cosseno(q, d)), decreasing = TRUE)

In [11]:
ix_aula <- montar(docs_aula(), cfg_aula())

q_aula <- vetor_consulta(tokenizar("modelo de recuperacao"),
                         ix_aula$vocab, ix_aula$idf_tfidf)

names(ranking_vetor(rocchio(q_aula, c("d2", "d3"), "d4", ix_aula), ix_aula))

[1] "d3" "d2" "d1" "d6" "d8" "d5" "d7" "d4"

In [12]:
q    <- usar[1]                                   # q01: monumentos historicos centro
g    <- qrels[[q]]                                # graus julgados de q01

q0   <- vetor_consulta(preparar(consultas[[q]], cfg),
                       ix$vocab, ix$idf_tfidf)    # vetor da consulta original

base <- ranking_vetor(q0, ix)                     # ranking sem Rocchio

Dr  <- intersect(names(g)[g >= cfg$limiar], names(docs))  # relevantes no corpus
Dnr <- intersect(names(base)[1:5], names(g)[g == 0])      # grau 0 visto no top-5

Dr
Dnr

[1] "d43"

character(0)

In [13]:
novo <- ranking_vetor(rocchio(q0, Dr, Dnr, ix), ix)

round(rbind(base    = metricas(base, g, cfg$limiar),
            rocchio = metricas(novo, g, cfg$limiar)), 3)

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0.000,0.023,0.023,0.183,0.183
rocchio,0.333,1.000,1.000,1.000,1.000


In [14]:
residual <- function(rk, g, fora, limiar)
  metricas(rk[!names(rk) %in% fora],
           g[!names(g) %in% fora], limiar)

fora <- c(Dr, Dnr)

round(rbind(base    = residual(base, g, fora, cfg$limiar),
            rocchio = residual(novo, g, fora, cfg$limiar)), 3)

,P_at_k,AP,RR,nDCG_bin,nDCG_grad
base,0,NA,0,NA,NA
rocchio,0,NA,0,NA,NA


In [15]:
Dr_v <- intersect(names(base)[1:5], Dr)
Dr_v

character(0)

In [17]:
# PRF: os top-k do ranking base viram Dr; depois, Rocchio e nova busca
prf <- function(q0, k, ix, p = c(a = 1, b = 0.75, g = 0.15)) {
  top <- names(ranking_vetor(q0, ix))[1:k]          # os k primeiros do ranking base
  ranking_vetor(rocchio(q0, top, ix = ix,
                        a = p[["a"]], b = p[["b"]], g = p[["g"]]), ix)
}

# AP de uma consulta; k = 0 significa sem PRF
ap_q <- function(txt, grau, k, ix, cfg) {
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
  rk <- if (k == 0) ranking_vetor(q0, ix) else prf(q0, k, ix)
  metricas(rk, grau, cfg$limiar)[["AP"]]
}

In [18]:
dev <- intersect(usar, nec$consulta[nec$conjunto == "dev"])

tab_dev <- sapply(dev, function(q) {
  a <- function(k) ap_q(consultas[[q]], qrels[[q]], k, ix, cfg)
  c(base = a(0), k2 = a(2), k3 = a(3), k5 = a(5))
})

round(tab_dev, 3)
round(rowMeans(tab_dev), 3)

,q01,q02,q03
base,0.023,0.074,0.079
k2,0.022,0.058,0.056
k3,0.019,0.083,0.055
k5,0.020,0.070,0.055


base    k2    k3    k5 
0.059 0.045 0.052 0.048

In [19]:
colnames(tab_dev)[tab_dev["k3", ] < tab_dev["base", ]]

qrels[["q01"]][names(ranking_cosseno(consultas[["q01"]], ix, cfg))[1:3]]
qrels[["q03"]][names(ranking_cosseno(consultas[["q03"]], ix, cfg))[1:3]]

[1] "q01" "q03"

<NA> <NA> <NA> 
  NA   NA   NA

<NA> <NA> <NA> 
  NA   NA   NA

In [20]:
source("config.R")

cfg$prf_k
cfg$rocchio

[1] 0

a    b    g 
1.00 0.75 0.15

In [21]:
ranking_motor <- function(txt, ix, cfg) {
  q0 <- vetor_consulta(preparar(txt, cfg), ix$vocab, ix$idf_tfidf)
  if (cfg$prf_k == 0) ranking_vetor(q0, ix) else prf(q0, cfg$prf_k, ix, cfg$rocchio)
}

round(head(ranking_motor(consultas[[usar[1]]], ix, cfg), 5), 3)

d6    d2    d4    d1    d3 
0.113 0.063 0.050 0.000 0.000

In [22]:
anexar_estado("00_FICHA_PROJETO.md")
anexar_estado("aula06_parteD_consolidado.md")

ERROR: Error: arquivo não encontrado: 00_FICHA_PROJETO.md
Salve primeiro o consolidado que a tutora gerou, depois rode de novo.
Você está em: /content


In [24]:
anexar_estado("00_FICHA_PROJETO.md")
anexar_estado("aula06_parteD_consolidado.md")

Estado do R anexado a 00_FICHA_PROJETO.md (62 linhas de objetos).

Estado do R anexado a aula06_parteD_consolidado.md (62 linhas de objetos).

